# 🎓 AWS BEDROCK LAB: Invoke Bedrock model for text generation using zero-shot prompt

## WHAT YOU'LL LEARN:
- How to connect to AWS Bedrock
- How to send prompts to Amazon Nova AI models
- Difference between non-streaming and streaming responses
- How to handle AI model responses

## KEY CONCEPTS:
1. **Non-streaming**: Get complete response all at once (like downloading a file)
2. **Streaming**: Get response in real-time chunks (like streaming a video)

## 📚 PREREQUISITE:
Make sure you have AWS credentials configured and access to Amazon Bedrock service.

## Introduction
You are Swami a Customer Service Manager at AnyCompany and some of your customers are not happy with the customer service and are providing negative feedbacks on the service provided by customer support engineers. Now, you would like to respond to those customers humbly aplogizing for the poor service and regain trust. You need the help of an LLM to generate a bulk of emails for you which are human friendly and personalized to the customer's sentiment from previous email correspondence.



## 🔧 SETUP: Import Libraries and Initialize Bedrock Client

This cell sets up everything we need to work with AWS Bedrock:
- We import necessary libraries (json, boto3, etc.)
- We configure AWS credentials and region settings
- We create a Bedrock client that will communicate with AWS AI models

Think of this as "opening the door" to AWS Bedrock services. The `boto3_bedrock` 
client is your connection to Amazon's AI models.

💡 **Tip**: If you get authentication errors, uncomment and fill in the AWS configuration 
lines with your region and credentials.

In [1]:
import json
import os
import sys
import boto3
import botocore

module_path = ".."
sys.path.append(os.path.abspath(module_path))
from labutils import bedrock, print_ww

# ---- ⚠️ Un-comment and edit the below lines as needed for your AWS setup ⚠️ ----
# os.environ["AWS_DEFAULT_REGION"] = "<REGION_NAME>"  # E.g. "us-east-1"
# os.environ["AWS_PROFILE"] = "<YOUR_PROFILE>"
# os.environ["BEDROCK_ASSUME_ROLE"] = "<YOUR_ROLE_ARN>"  # E.g. "arn:aws:..."

boto3_bedrock = bedrock.get_bedrock_client(
    assumed_role=os.environ.get("BEDROCK_ASSUME_ROLE", None),
    region=os.environ.get("AWS_DEFAULT_REGION", None)
)

Create new client
  Using region: us-east-1
boto3 Bedrock client successfully created!
bedrock-runtime(https://bedrock-runtime.us-east-1.amazonaws.com)


## 📝 PROMPT PREPARATION: Create Your Request

Here we're preparing what we want to ask the AI model:
- `prompt_data`: Contains our instruction (write an email from Swami to John Doe)
- `body`: The complete request formatted for Amazon Nova models

### 🔑 Amazon Nova Request Structure:
Nova uses a "messages" format with:
- `"role": "user"` (who's asking)
- `"content": [{"text": your_prompt}]` (what you're asking)

### inferenceConfig controls how the AI responds:
- `max_new_tokens`: Maximum length of response (4096 tokens ≈ 3000 words)
- `temperature`: 0 means deterministic/consistent responses
- `topP`: 0.9 controls response diversity

`modelId` specifies we're using Amazon Nova Lite version 1.0

In [2]:
# create the prompt
prompt_data = """
Command: Write an email from Swami, Customer Service Manager, to the customer "John Doe" 
who provided negative feedback on the service provided by our customer support 
engineer"""

# Nova models use a different request format
body = json.dumps({
    "messages": [
        {
            "role": "user",
            "content": [{"text": prompt_data}]
        }
    ],
    "inferenceConfig": {
        "max_new_tokens": 4096,
        "temperature": 0,
        "topP": 0.9
    }
})

modelId = 'us.amazon.nova-2-lite-v1:0'
accept = 'application/json'
contentType = 'application/json'

## ⚡ NON-STREAMING INVOCATION: Get Complete Response at Once

This cell sends your request and waits for the COMPLETE response:
- `invoke_model()` sends the request and waits for full response
- We parse the JSON response to extract the AI's text

### 📊 Nova Response Structure:
```
response_body → output → message → content[0] → text
```

This is like sending an email and waiting for the complete reply before reading it.

### ✅ Use this when: 
You want the entire response before processing it

### ❌ Don't use when: 
Generating very long responses where you want to see progress

**Error handling** catches `AccessDeniedException` if you don't have proper AWS permissions.

In [3]:
outputText = "\n"

try:
    response = boto3_bedrock.invoke_model(body=body, modelId=modelId, accept=accept, contentType=contentType)
    response_body = json.loads(response.get('body').read())
    
    # Nova response format
    outputText = response_body.get('output', {}).get('message', {}).get('content', [{}])[0].get('text', '')
    
    print('\t\t\x1b[31m**Non-streaming Output**\x1b[0m\n')
    print(outputText)
    
except botocore.exceptions.ClientError as error:
    
    if error.response['Error']['Code'] == 'AccessDeniedException':
           print(f"\x1b[41m{error.response['Error']['Message']}\
                \nTo troubleshoot this issue please refer to the following resources.\
                 \nhttps://docs.aws.amazon.com/IAM/latest/UserGuide/troubleshoot_access-denied.html\
                 \nhttps://docs.aws.amazon.com/bedrock/latest/userguide/security-iam.html\x1b[0m\n")
        
    else:
        raise error

		**Non-streaming Output**

### **Subject: We’re Sorry – Your Feedback Matters**  

Dear John Doe,

I hope this message finds you well.

My name is **Swami**, and I am the **Customer Service Manager** at [Your Company Name]. I am writing to you personally in response to the feedback you recently shared regarding your experience with one of our customer support engineers. First and foremost, I want to sincerely apologize for the disappointment and frustration you encountered during your interaction. We deeply regret that your experience did not meet the standards we strive to uphold.

At [Your Company Name], we are committed to providing exceptional customer support, and it is clear that we fell short in your case. Your feedback is incredibly valuable to us, as it helps us identify areas where we can improve both our processes and the training we provide to our support team.

We have taken your concerns seriously and have initiated a thorough review of the interaction you described. Our

## 🌊 STREAMING INVOCATION: Get Response Piece by Piece

This cell uses streaming to receive the AI's response in real-time chunks:
- `invoke_model_with_response_stream()` starts receiving immediately
- Each "chunk" is a piece of the response as it's generated
- We print each chunk as it arrives

### 🎯 How Streaming Works:
1. Send request
2. AI starts generating text
3. Receive chunk 1 → print it
4. Receive chunk 2 → print it
5. Continue until complete

### Nova Streaming Format:
Each chunk contains: `contentBlockDelta → delta → text`

### ✅ Use this when: 
You want immediate feedback and live updates

### 📱 Perfect for: 
Chat interfaces, long responses, better user experience

This is like watching someone type a message in real-time rather than waiting 
for them to finish before seeing anything.

In [5]:
output = []
full_text = ""

try:
    response = boto3_bedrock.invoke_model_with_response_stream(
        body=body, modelId=modelId, accept=accept, contentType=contentType
    )
    stream = response.get('body')

    # Collect all streamed tokens first
    if stream:
        for event in stream:
            chunk = event.get('chunk')
            if chunk:
                chunk_obj = json.loads(chunk.get('bytes').decode())
                if 'contentBlockDelta' in chunk_obj:
                    text = chunk_obj['contentBlockDelta']['delta']['text']
                    full_text += text
                    output.append(text)

    # Split full text into exactly 10 chunks
    total_len = len(full_text)
    chunk_size = total_len // 10

    print("Streaming output in 10 chunks:\n")
    for i in range(10):
        start = i * chunk_size
        # Last chunk gets any remaining characters
        end = start + chunk_size if i < 9 else total_len
        chunk_text = full_text[start:end]
        print(f'\t\t\x1b[31m**Chunk {i+1}**\x1b[0m\n{chunk_text}\n')

except botocore.exceptions.ClientError as error:
    if error.response['Error']['Code'] == 'AccessDeniedException':
        print(f"\x1b[41m{error.response['Error']['Message']}\
             \nTo troubleshoot this issue please refer to the following resources.\
              \nhttps://docs.aws.amazon.com/IAM/latest/UserGuide/troubleshoot_access-denied.html\
              \nhttps://docs.aws.amazon.com/bedrock/latest/userguide/security-iam.html\x1b[0m\n")
    else:
        raise error

Streaming output in 10 chunks:

		**Chunk 1**
### **Subject: We’re Sorry – Your Feedback Matters**  

Dear John Doe,

I hope this message finds you well.

My name is **Swami**, and I am the **Customer Service Manager** at [Your Company Name]. I am writing to you personally in r

		**Chunk 2**
esponse to the feedback you recently shared regarding your experience with our customer support team. First and foremost, I want to sincerely apologize for the disappointment and frustration you encountered during your interaction w

		**Chunk 3**
ith our support engineer.

At [Your Company Name], we strive to provide every customer with prompt, professional, and effective support. It is clear that we fell short of our own standards in your case, and for that, I am truly sorr

		**Chunk 4**
y.

We take all feedback seriously, especially when it highlights areas where we can improve. Your comments have been carefully reviewed, and I have personally shared them with our support team lead and trainin

## 📋 FINAL OUTPUT: Display Complete Streamed Response

This cell combines all the chunks we received during streaming:
- `''.join(output)` combines all text chunks into one complete string
- Prints the full response for easy reading

### Why do we need this?
During streaming (Cell 4), chunks are printed separately. This cell shows you 
the final, complete response all together - useful for copying or reviewing 
the entire output.

**Think of it like**: Cell 4 shows you the movie frame-by-frame, Cell 5 shows you 
the complete film.

In [6]:
# The relevant portion of the response begins after the first newline character
# Below we print the response beginning after the first occurence of '\n'.

email = outputText[outputText.index('\n')+1:]
print_ww(email)


Dear John Doe,

I hope this message finds you well.

My name is **Swami**, and I am the **Customer Service Manager** at [Your Company Name]. I am writing
to you personally in response to the feedback you recently shared regarding your experience with one
of our customer support engineers. First and foremost, I want to sincerely apologize for the
disappointment and frustration you encountered during your interaction. We deeply regret that your
experience did not meet the standards we strive to uphold.

At [Your Company Name], we are committed to providing exceptional customer support, and it is clear
that we fell short in your case. Your feedback is incredibly valuable to us, as it helps us identify
areas where we can improve both our processes and the training we provide to our support team.

We have taken your concerns seriously and have initiated a thorough review of the interaction you
described. Our support engineer has been counseled on the importance of active listening, empathy

In [7]:
print('\t\t\x1b[31m**COMPLETE OUTPUT**\x1b[0m\n')
complete_output = ''.join(output)
print(complete_output)

		**COMPLETE OUTPUT**

### **Subject: We’re Sorry – Your Feedback Matters**  

Dear John Doe,

I hope this message finds you well.

My name is **Swami**, and I am the **Customer Service Manager** at [Your Company Name]. I am writing to you personally in response to the feedback you recently shared regarding your experience with our customer support team. First and foremost, I want to sincerely apologize for the disappointment and frustration you encountered during your interaction with our support engineer.

At [Your Company Name], we strive to provide every customer with prompt, professional, and effective support. It is clear that we fell short of our own standards in your case, and for that, I am truly sorry.

We take all feedback seriously, especially when it highlights areas where we can improve. Your comments have been carefully reviewed, and I have personally shared them with our support team lead and training department. This will help us identify specific gaps in our processes

### Complete output

In [8]:
print('\t\t\x1b[31m**COMPLETE OUTPUT**\x1b[0m\n')
complete_output = ''.join(output)
print(complete_output)

		**COMPLETE OUTPUT**

### **Subject: We’re Sorry – Your Feedback Matters**  

Dear John Doe,

I hope this message finds you well.

My name is **Swami**, and I am the **Customer Service Manager** at [Your Company Name]. I am writing to you personally in response to the feedback you recently shared regarding your experience with our customer support team. First and foremost, I want to sincerely apologize for the disappointment and frustration you encountered during your interaction with our support engineer.

At [Your Company Name], we strive to provide every customer with prompt, professional, and effective support. It is clear that we fell short of our own standards in your case, and for that, I am truly sorry.

We take all feedback seriously, especially when it highlights areas where we can improve. Your comments have been carefully reviewed, and I have personally shared them with our support team lead and training department. This will help us identify specific gaps in our processes